# Exercício Prático — Aula 08: Web Scraping com BeautifulSoup

**Análise e Ciência de Dados** (Especialização, IFRN) — Unidade 2, Aula 08

**Entrega:** este notebook **executado** (com as saídas visíveis) + os arquivos CSV/JSON gerados, compactados em um `.zip`, na tarefa *Exercício Prático — Aula 08* do Moodle. **Prazo: 24/10/2026 (23h59).**

Todos os trechos marcados com `# TODO` foram preenchidos e comentados. Use o notebook `Aula08_Web_Scraping_BeautifulSoup.ipynb` como referência — os padrões de código são os mesmos.
Coloque este notebook na mesma pasta da subpasta `site_resenhas/` (vem no pacote da aula).

In [57]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time, json, sys, subprocess
from io import StringIO
from urllib.parse import urljoin
from pathlib import Path
from urllib.robotparser import RobotFileParser

SAIDA = Path("dados_exercicio")
SAIDA.mkdir(exist_ok=True)
CABECALHOS = {"User-Agent": "ALEXANDRE-CLEMENTE-AulaACD/1.0 (exercicio educacional)"}
pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 12)

def baixar(url):
    resp = requests.get(url, headers=CABECALHOS, timeout=10)
    resp.raise_for_status()
    resp.encoding = "utf-8"
    return BeautifulSoup(resp.text, "html.parser")

## Parte A — Estante Potiguar (site local)

1. Coloque o site no ar (célula abaixo) e raspe **todas as resenhas**, seguindo o link "Próxima" (`li.proxima a`) e esperando 1 s entre as páginas.
2. Monte um DataFrame com: `livro`, `autor`, `nota`, `leitor`, `data`, `comentario`, `util` (use `None` quando o contador não existir).

In [58]:
servidor = subprocess.Popen([sys.executable, "-m", "http.server", "8000", "--directory", "site_resenhas"],
                            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1.5)
BASE = "http://localhost:8000/"

url = BASE

# Lendo o robots.txt
robo = RobotFileParser(f"{BASE}/robots.txt")
robo.read()

print("\nrobots.txt:")
for caminho in ["index.html", "ranking.html", "privado/rascunhos.html"]:
    print(f"{caminho:25s} permitido? {robo.can_fetch(CABECALHOS['User-Agent'], urljoin(BASE, caminho))}")
print("Intervalo pedido entre requisições (Crawl-delay):", robo.crawl_delay("*"), "s")

print("---")

resposta = requests.get(BASE, headers=CABECALHOS, timeout=10)
print("Status:", resposta.status_code)
print("Content-Type:", resposta.headers.get("Content-Type"))
print("Encoding adivinhado pelo requests:", resposta.encoding)
print("Título lido sem corrigir:", BeautifulSoup(resposta.text, "html.parser").title.get_text())

print("---")

resposta.encoding = "utf-8"
pagina = BeautifulSoup(resposta.text, "html.parser")
print("Título corrigido:", pagina.title.get_text())
print("Resenhas nesta página:", len(pagina.select("article.resenha")))

print("---")

# baixar a página

def baixar(url):
    resp = requests.get(url, headers=CABECALHOS, timeout=10)
    resp.raise_for_status()
    resp.encoding = "utf-8"
    return resp

# extrair cada article.resenha
    
def extrair_resenhas(soup):
    registros = []
    for art in soup.select("article.resenha"):
        util = art.select_one(".util")
        registros.append({
            "id": int(art["data-id"]),
            "livro": art.select_one("h2.livro").get_text(strip=True),
            "autor": art.select_one("p.autor").get_text(strip=True),
            "nota": int(art.select_one("div.nota")["data-nota"]),
            "leitor": art.select_one("span.nome").get_text(strip=True),
            "data": art.select_one("time")["datetime"],
            "comentario": art.select_one("p.comentario").get_text(strip=True),
            "util": int(util.get_text().split()[0]) if util else None,
        })
    return registros

# Laço para paginação e extrair todas as páginas
todas = []
pausa = robo.crawl_delay("*") or 1
resenhas = []

while url:
    soup_pag = BeautifulSoup(baixar(url).text, "html.parser")
    novas = extrair_resenhas(soup_pag)
    resenhas.extend(novas)
    print(f"{url:40s} -> {len(novas)} resenhas (acumulado: {len(todas)})")

    link = soup_pag.select_one("li.proxima a")
    url = urljoin(url, link["href"]) if link else None
    if url:
        time.sleep(pausa)

df_resenhas = pd.DataFrame(resenhas)
print("\nTotal:", len(df_resenhas), "resenhas")


robots.txt:
index.html                permitido? True
ranking.html              permitido? True
privado/rascunhos.html    permitido? False
Intervalo pedido entre requisições (Crawl-delay): 1 s
---
Status: 200
Content-Type: text/html
Encoding adivinhado pelo requests: ISO-8859-1
Título lido sem corrigir: Resenhas â pÃ¡gina 1 | Estante Potiguar
---
Título corrigido: Resenhas — página 1 | Estante Potiguar
Resenhas nesta página: 8
---
http://localhost:8000/                   -> 8 resenhas (acumulado: 0)
http://localhost:8000/pagina-2.html      -> 8 resenhas (acumulado: 0)
http://localhost:8000/pagina-3.html      -> 8 resenhas (acumulado: 0)

Total: 24 resenhas


3. Calcule, **por autor**, o número de resenhas e a nota média (ordenado da maior para a menor média).
4. Mostre a resenha **mais útil** (maior valor de `util`): livro, leitor e quantidade.
5. Leia a tabela `table#ranking` de `ranking.html` com `pd.read_html(..., decimal=",")` e confira se as médias por livro batem com as que você calcularia a partir das resenhas.

In [59]:
# 3. agrupamento por autor

resumo_autor = (df_resenhas.groupby("autor")["nota"]
          .agg(resenhas="count", media="mean")
          .sort_values("media", ascending=False)
          .rename(columns={"autor": "AUTOR", "resenhas": "QTD_RESENHAS", "media": "NOTA MÉDIA"}))

print("\n### AGREGAÇÃO POR AUTOR COM TOTAL DE RESENHAS E MÉDIA DAS NOTAS ###")
print(resumo_autor.round(2))
print("---")

# 4. resenha mais útil

maior_util = df_resenhas["util"].max()
resenhas_mais_uteis = df_resenhas[df_resenhas["util"].eq(maior_util)]

print("\n### RESENHA MAIS ÚTIL (QTD DE CURTIDAS) ###")
print(resenhas_mais_uteis[["livro", "leitor", "comentario", "util"]].to_string(index=False))
print("---")

# agrupamento por livro e comparação

resumo_livro = (df_resenhas.groupby("livro")["nota"]
          .agg(resenhas="count", media_calculada="mean")
          .sort_values("media_calculada", ascending=False)
          .rename(columns={"resenhas": "QTD_RESENHAS", "media_calculada": "NOTA MÉDIA"}))

# print("\n### AGREGAÇÃO POR LIVRO COM TOTAL DE RESENHAS E MÉDIA DAS NOTAS ###")
# print(resumo_livro)
# print("---")

# 5. tabela de ranking com pd.read_html e comparação

resp = baixar(urljoin(BASE, "ranking.html"))

ranking = pd.read_html(StringIO(resp.text), attrs={"id": "ranking"}, decimal=",", thousands=".")[0]
# print("\n### RANKING TABLE ###")
# print(ranking)
# print()
# print(ranking.dtypes)
ranking_rename = ranking.rename(columns={"Resenhas": "QTD_RESENHAS", "Nota média": "MÉDIA TABELA"})

# print("\n### RANKING RENAME TABLE ###")
# print(ranking_rename[["Livro", "QTD_RESENHAS", "MÉDIA TABELA"]])
# print("---")

print("\n### COMPARAÇÃO MÉDIA CALCULADA E MÉDIA DA TABELA RANKING DO SITE ###")
comparacao = resumo_livro.reset_index().merge(
    ranking_rename[["Livro", "MÉDIA TABELA"]], 
    left_on="livro",
    right_on="Livro",
    how="left"
).drop(columns=["QTD_RESENHAS", "Livro"])

print(comparacao)



### AGREGAÇÃO POR AUTOR COM TOTAL DE RESENHAS E MÉDIA DAS NOTAS ###
                  QTD_RESENHAS  NOTA MÉDIA
autor                                     
Machado de Assis            12        4.33
Aluísio Azevedo              4        4.00
Lima Barreto                 4        4.00
José de Alencar              4        3.50
---

### RESENHA MAIS ÚTIL (QTD DE CURTIDAS) ###
       livro leitor                                                                                                       comentario  util
Dom Casmurro Ana P. Releio a cada cinco anos e sempre mudo de opinião sobre Capitu. Narrador mais suspeito da literatura brasileira.  31.0
---

### COMPARAÇÃO MÉDIA CALCULADA E MÉDIA DA TABELA RANKING DO SITE ###
                              livro  NOTA MÉDIA  MÉDIA TABELA
0                       O Alienista        4.50          4.50
1                      Dom Casmurro        4.25          4.25
2   Memórias Póstumas de Brás Cubas        4.25          4.25
3                       

### Conclusão

Nota-se com a impressão da comparação das média da tabela acima "**### COMPARAÇÃO MÉDIA CALCULADA E MÉDIA DA TABELA RANKING DO SITE ###**", que os valores são consistentes, confirmando a precisão do scraping realizado.

## Parte B — Books to Scrape (site real de treino)

6. Escolha **uma categoria com pelo menos 2 páginas** (lista no enunciado da tarefa; ex.: *Mystery*, *Romance*, *Fantasy*, *Food and Drink*). Os links das categorias estão em `div.side_categories ul li ul li a` na página inicial.
7. Raspe **todos os livros da categoria**, seguindo `li.next a` até o fim e esperando 1 s entre as páginas. Para cada livro guarde: `titulo` (atributo `title`), `preco` (float, sem o £), `nota` (1 a 5, a partir da classe `star-rating`) e `link` **absoluto** (use `urljoin` — nas categorias o `href` começa com `../../../`).

In [60]:
# Função para encontrar o link da categoria
def encontrar_url_categoria(index_html):
    link_tag = index_html.find("div", class_="side_categories").find(
        "a", string=lambda t: t and t.strip().lower() == CATEGORIA.lower()
    )
    url_categoria = urljoin(BASE, link_tag["href"])
    print(url_categoria)
    return url_categoria

# Função para extrair os livros da página html

def extrair_livros(pagina_html):
    livros = []
    for pod in pagina_html.select("article.product_pod"):
        classe_nota = pod.select_one("p.star-rating")["class"][1]      # ex.: "Three"
        livros.append({
            "titulo": pod.select_one("h3 a")["title"],
            "preco_libras": float(pod.select_one("p.price_color").get_text().replace("£", "")),
            "nota": NOTAS[classe_nota],
        })
    return livros

BASE = "https://books.toscrape.com"
CATEGORIA = "Mystery"   # escolha a sua
NOTAS = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

# encontrar a url da CATEGORIA na barra lateral
resp = baixar(f"{BASE}/index.html")
index_html = BeautifulSoup(resp.text, "html.parser")
url = encontrar_url_categoria(index_html)

# laço de paginação na categoria (li.next a), com time.sleep(1)
livros = []
paginas_lidas = 0

while url and paginas_lidas < 5:
    soup_pag = BeautifulSoup(baixar(url).text, "html.parser")
    livros.extend(extrair_livros(soup_pag))
    paginas_lidas += 1
    print(soup_pag.select_one("li.current").get_text(strip=True), "->", len(livros), "livros")

    proxima = soup_pag.select_one("li.next a")
    url = urljoin(url, proxima["href"]) if proxima else None
    time.sleep(1)

https://books.toscrape.com/catalogue/category/books/mystery_3/index.html
Page 1 of 2 -> 20 livros
Page 2 of 2 -> 32 livros


8. Calcule: total de livros, preço médio, distribuição das notas e os **3 livros mais bem avaliados** (desempate pelo **menor** preço).
9. Abra a página de detalhe de **um** desses livros e extraia a tabela `table.table-striped` em um dicionário.

In [61]:
# estatísticas da categoria
print("### ESTATÍSTICAS ###")
df_livros = pd.DataFrame(livros)
print("Total: ", len(df_livros), "livros na categoria", CATEGORIA)
print("\nPreço médio: £", round(df_livros["preco_libras"].mean(), 2))
print()
print(df_livros.groupby("nota")["preco_libras"].agg(["count", "mean"]).round(2).rename(columns={"count": "QTD Notas", "mean": "Preço Médio"}))
print("\n### TOP 3 MAIS BEM AVALIADOS ORDENADOS POR MENOR PREÇO ###")
top3_avaliados = (
    df_livros
    .sort_values(by=["nota", "preco_libras"], ascending=[False, True])
    .head(3)
)
print(top3_avaliados)

# tabela da página de detalhe de um livro
url_livro = "https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html"
soup_livro = BeautifulSoup(baixar(url_livro).text, "html.parser")
print("\n### Conteúdo de página de um livro ###")
print("\nTítulo:", soup_livro.select_one("div.product_main h1").get_text())
print("Categoria:", soup_livro.select("ul.breadcrumb li")[2].get_text(strip=True))
info = {tr.th.get_text(strip=True): tr.td.get_text(strip=True)
        for tr in soup_livro.select("table.table-striped tr")}
for chave, valor in info.items():
    print(f"  {chave:20s} {valor}")


### ESTATÍSTICAS ###
Total:  32 livros na categoria Mystery

Preço médio: £ 31.72

      QTD Notas  Preço Médio
nota                        
1             7        20.84
2             5        32.26
3             8        30.03
4             7        43.74
5             5        32.27

### TOP 3 MAIS BEM AVALIADOS ORDENADOS POR MENOR PREÇO ###
                                               titulo  preco_libras  nota
28                                  The Girl You Lost         12.29     5
22                  The Silkworm (Cormoran Strike #2)         23.05     5
17  What Happened on Beale Street (Secrets of the ...         25.37     5

### Conteúdo de página de um livro ###

Título: A Light in the Attic
Categoria: Poetry
  UPC                  a897fe39b1053632
  Product Type         Books
  Price (excl. tax)    £51.77
  Price (incl. tax)    £51.77
  Tax                  £0.00
  Availability         In stock (22 available)
  Number of reviews    0


## Parte C — Salvar e refletir

10. Salve em `dados_exercicio/`: as resenhas (CSV) e os livros da categoria (CSV + JSON com **proveniência**: fonte, URL, data da coleta).
11. Responda na célula Markdown abaixo (3 a 5 linhas): que cuidados éticos e legais você tomou (robots.txt, identificação, intervalo entre requisições, dados pessoais/LGPD)? O que mudaria se o site **não** permitisse a coleta?

In [62]:
# salvar CSV/JSON
DADOS = Path("dados")
DADOS.mkdir(exist_ok=True)
df_resenhas.to_csv(DADOS / "resenhas_estante_potiguar.csv", index=False, encoding="utf-8")
df_livros.to_csv(DADOS / "books_toscrape_5paginas.csv", index=False, encoding="utf-8")

with open(DADOS / "resenhas_estante_potiguar.json", "w", encoding="utf-8") as f:
    json.dump({
        "fonte": "Estante Potiguar (site fictício da Aula 08, servido localmente)",
        "url": BASE,
        "coletado_em": "2026-10-09",
        "observacao": "leitores e comentários fictícios; nomes abreviados",
        "resenhas": todas,
    }, f, ensure_ascii=False, indent=2)

for arq in sorted(DADOS.glob("*")):
    if not arq.name.startswith("_"):
        print(arq.name)
servidor.terminate()

books_toscrape_5paginas.csv
resenhas_estante_potiguar.csv
resenhas_estante_potiguar.json


**Reflexão ética (escreva aqui):**

...

**Reflexão ética:** Consultei o `robots.txt` do site local, respeitei as restrições indicadas e usei um `User-Agent` identificando a finalidade educacional da coleta. Mantive um intervalo de 1 segundo entre requisições para reduzir a carga no servidor e coletei apenas dados públicos necessários ao exercício, evitando informações pessoais sensíveis conforme a LGPD. Se o site não permitisse a coleta, eu interromperia o scraping e buscaria autorização, uma API oficial ou uma fonte alternativa autorizada.